# Session 19 — NLP Fundamentals: Text Processing in Practice

Welcome to the hands-on half of Session 19! In the slides we learned **what** happens to text before a model sees it.
Now we'll do every step ourselves, in code.

**What we'll build today:**

1. **Tokenization** — splitting text into words, sentences, and characters
2. **Vocabulary** — mapping tokens to ID numbers (and meeting the `[UNK]` problem)
3. **Stop words** — filtering noise (and learning when *not* to)
4. **Stemming** — fast, crude word chopping with the Porter Stemmer
5. **Lemmatization** — dictionary-correct base forms with WordNet
6. **N-grams** — capturing word order (we'll build a tiny autocomplete!)
7. **The full pipeline** — everything combined into one function
8. **Bonus:** a real BERT tokenizer — subwords in production

> **How to use this notebook:** run each cell with `Shift+Enter`. Read the markdown before each code cell — it explains *why* before the code shows *how*.

## 0. Setup

We use **NLTK** (Natural Language Toolkit) — the classic Python library for text processing.
NLTK keeps its language data (tokenizer models, stop word lists, the WordNet dictionary) in separate downloadable packages, so we fetch those too. You only need to run the downloads once per machine.

In [1]:
# Install NLTK if you don't have it (uncomment on first run)
# !pip install nltk

import nltk

# Download the data packages we need (quiet=True hides the progress logs)
nltk.download('punkt', quiet=True)        # sentence & word tokenizer models
nltk.download('punkt_tab', quiet=True)    # newer NLTK versions need this too
nltk.download('stopwords', quiet=True)    # stop word lists for 16 languages
nltk.download('wordnet', quiet=True)      # the dictionary used by the lemmatizer
nltk.download('omw-1.4', quiet=True)      # multilingual wordnet data
nltk.download('averaged_perceptron_tagger_eng', quiet=True)  # part-of-speech tagger

print("Setup complete! NLTK version:", nltk.__version__)

Setup complete! NLTK version: 3.9.4


---
## 1. Tokenization

A **token** is the smallest unit of text a model works with. **Tokenization** = splitting text into tokens.

Let's start with the most naive approach — Python's built-in `.split()` — and see why it isn't good enough.

In [2]:
text = "Don't panic! Tokenization isn't hard. It's the first step of every NLP system."

# Naive approach: split on spaces
naive_tokens = text.split()
print("Naive split:", naive_tokens)
print("Number of tokens:", len(naive_tokens))

Naive split: ["Don't", 'panic!', 'Tokenization', "isn't", 'hard.', "It's", 'the', 'first', 'step', 'of', 'every', 'NLP', 'system.']
Number of tokens: 13


Look closely at the output: `"panic!"` and `"hard."` keep their punctuation glued on.
That means `"hard"` and `"hard."` would be **two different tokens** — exactly the kind of mess we want to avoid.

NLTK's `word_tokenize` is smarter: it separates punctuation and even understands contractions.

In [3]:
from nltk.tokenize import word_tokenize

tokens = word_tokenize(text)
print("NLTK tokens:", tokens)
print("Number of tokens:", len(tokens))

NLTK tokens: ['Do', "n't", 'panic', '!', 'Tokenization', 'is', "n't", 'hard', '.', 'It', "'s", 'the', 'first', 'step', 'of', 'every', 'NLP', 'system', '.']
Number of tokens: 19


Notice two things:

* Punctuation (`!`, `.`) became **separate tokens** — `hard` is now just `hard`.
* `Don't` was split into `Do` + `n't` — NLTK knows a contraction hides two words ("do" + "not").

### Sentence tokenization

Sometimes the unit we want isn't a word but a **sentence** (e.g., for summarization). Note that
`sent_tokenize` is smart enough not to break on the dot in "Dr." — a naive split on `"."` would fail there.

In [4]:
from nltk.tokenize import sent_tokenize

paragraph = ("Dr. Smith teaches NLP at the university. "
             "Her favorite topic is tokenization. "
             "Students love the practical sessions!")

sentences = sent_tokenize(paragraph)
for i, s in enumerate(sentences, 1):
    print(f"Sentence {i}: {s}")

Sentence 1: Dr. Smith teaches NLP at the university.
Sentence 2: Her favorite topic is tokenization.
Sentence 3: Students love the practical sessions!


### Character tokenization

The other extreme: every character is a token. Trivial to implement, never meets an unknown symbol —
but look how long the sequence gets for a single word.

In [5]:
word = "unhappiness"

char_tokens = list(word)
print("Character tokens:", char_tokens)
print(f"1 word became {len(char_tokens)} tokens!")

# Compare the three granularities from the slides:
print()
print("Word-level:     ['unhappiness']                  -> 1 token")
print("Subword-level:  ['un', 'happi', 'ness']          -> 3 tokens (we'll see a real one in the bonus!)")
print(f"Character-level: {char_tokens} -> {len(char_tokens)} tokens")

Character tokens: ['u', 'n', 'h', 'a', 'p', 'p', 'i', 'n', 'e', 's', 's']
1 word became 11 tokens!

Word-level:     ['unhappiness']                  -> 1 token
Subword-level:  ['un', 'happi', 'ness']          -> 3 tokens (we'll see a real one in the bonus!)
Character-level: ['u', 'n', 'h', 'a', 'p', 'p', 'i', 'n', 'e', 's', 's'] -> 11 tokens


**Trade-off recap:** words = short sequences but huge vocabulary; characters = tiny vocabulary but very long sequences;
**subwords** (used by GPT & BERT) sit in the sweet spot. We'll play with a real subword tokenizer at the end.

---
## 2. Vocabulary: from tokens to numbers

Models don't eat strings — they eat **numbers**. The **vocabulary** is the fixed list of all tokens a model knows,
each mapped to a unique ID. Let's build one from scratch from a mini-corpus.

In [6]:
mini_corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "a cat and a dog played",
]

# Step 1: tokenize every sentence and collect unique tokens
all_tokens = []
for sentence in mini_corpus:
    all_tokens.extend(word_tokenize(sentence))

unique_tokens = sorted(set(all_tokens))
print("Unique tokens in corpus:", unique_tokens)

Unique tokens in corpus: ['a', 'and', 'cat', 'dog', 'mat', 'on', 'played', 'rug', 'sat', 'the']


In [7]:
# Step 2: build the vocabulary — special tokens first, then real words
vocab = {"[PAD]": 0, "[UNK]": 1}          # 0 = padding, 1 = unknown
for token in unique_tokens:
    vocab[token] = len(vocab)             # next free ID

print("Vocabulary (token -> ID):")
for token, idx in vocab.items():
    print(f"  {token!r:10} -> {idx}")
print("Vocabulary size:", len(vocab))

Vocabulary (token -> ID):
  '[PAD]'    -> 0
  '[UNK]'    -> 1
  'a'        -> 2
  'and'      -> 3
  'cat'      -> 4
  'dog'      -> 5
  'mat'      -> 6
  'on'       -> 7
  'played'   -> 8
  'rug'      -> 9
  'sat'      -> 10
  'the'      -> 11
Vocabulary size: 12


Now we can **encode** any sentence: look every token up and return its ID.
Tokens that are *not* in the vocabulary get the `[UNK]` ID — and that's where information dies.

In [8]:
def encode(sentence, vocab):
    """Convert a sentence into a list of vocabulary IDs."""
    tokens = word_tokenize(sentence.lower())
    return [vocab.get(token, vocab["[UNK]"]) for token in tokens]  # .get -> [UNK] if missing

# A sentence made only of known words:
print("'the cat sat'        ->", encode("the cat sat", vocab))

# A sentence with words the vocabulary has never seen:
sentence = "the elephant danced"
ids = encode(sentence, vocab)
print(f"'{sentence}' ->", ids)
print("Notice the 1s: 'elephant' and 'danced' both collapsed to [UNK] — their meaning is lost!")

'the cat sat'        -> [11, 4, 10]
'the elephant danced' -> [11, 1, 1]
Notice the 1s: 'elephant' and 'danced' both collapsed to [UNK] — their meaning is lost!


This is the **out-of-vocabulary (OOV) problem** from the slides. Classic word-level NLP suffers from it badly;
subword tokenizers (bonus section) were invented to make it disappear.

---
## 3. Stop words: filtering the noise

**Stop words** are extremely common words (*the, is, a, in...*) that often add little meaning.
NLTK ships ready-made lists for 16 languages.

In [9]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print(f"NLTK knows {len(stop_words)} English stop words.")
print("A sample:", sorted(list(stop_words))[:15])

NLTK knows 198 English stop words.
A sample: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]


In [10]:
sentence = "The quick brown fox jumps over the lazy dog"

tokens = word_tokenize(sentence.lower())
filtered = [t for t in tokens if t not in stop_words]

print("Before:", tokens)
print("After: ", filtered)
print(f"\nWe dropped {len(tokens) - len(filtered)} of {len(tokens)} tokens and kept the meaning!")

Before: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
After:  ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']

We dropped 3 of 9 tokens and kept the meaning!


### ⚠️ The trap: stop word removal can flip meaning

Stop word lists include the word **"not"**. Watch what happens to a negative movie review:

In [11]:
review = "The movie was not good"

tokens = word_tokenize(review.lower())
filtered = [t for t in tokens if t not in stop_words]

print("Original:        ", review)
print("After filtering: ", filtered)
print()
print("'not' is a stop word ->", 'not' in stop_words)
print("A negative review just became a POSITIVE one. Removal is a choice, not a rule!")

Original:         The movie was not good
After filtering:  ['movie', 'good']

'not' is a stop word -> True
A negative review just became a POSITIVE one. Removal is a choice, not a rule!


**Rule of thumb:** remove stop words for search / keyword extraction / topic modeling;
**keep them** for sentiment analysis, translation, chatbots — anywhere grammar and negation matter.

---
## 4. Stemming: the fast axe

**Stemming** chops word endings with hard-coded rules to reach a common "stem". The classic is the **Porter Stemmer** (1980).
It's fast — and crude: the output is often not a real word.

In [12]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

words = ["running", "runs", "ran", "studies", "studying", "likes", "likely", "liked", "history", "historical"]

print(f"{'word':<12} -> stem")
print("-" * 25)
for w in words:
    print(f"{w:<12} -> {stemmer.stem(w)}")

word         -> stem
-------------------------
running      -> run
runs         -> run
ran          -> ran
studies      -> studi
studying     -> studi
likes        -> like
likely       -> like
liked        -> like
history      -> histori
historical   -> histor


Observations (match these to the slides!):

* `running`, `runs` → `run` ✓ — but `ran` stays `ran` (irregular forms slip through, stemming has no dictionary)
* `studies` → `studi` — **not an English word** at all
* `likely` → `like` — an adverb collapsed into a verb

### Over-stemming and under-stemming

In [13]:
# OVER-stemming: unrelated meanings collapse to the same stem (false positives)
over = ["university", "universal", "universe"]
print("Over-stemming  :", {w: stemmer.stem(w) for w in over})
print("  -> three different concepts, ONE stem. Search for 'universe' would match 'university'!\n")

# UNDER-stemming: clearly related words FAIL to unite (false negatives)
under = ["alumnus", "alumnae", "alumni"]
print("Under-stemming :", {w: stemmer.stem(w) for w in under})
print("  -> same family, three different stems. They should have matched!")

Over-stemming  : {'university': 'univers', 'universal': 'univers', 'universe': 'univers'}
  -> three different concepts, ONE stem. Search for 'universe' would match 'university'!

Under-stemming : {'alumnus': 'alumnu', 'alumnae': 'alumna', 'alumni': 'alumni'}
  -> same family, three different stems. They should have matched!


---
## 5. Lemmatization: the precise scalpel

**Lemmatization** looks words up in a dictionary (WordNet) and returns the proper base form — the **lemma**.
The output is always a real word, but it needs to know the **part of speech** (POS) to do its best work.

In [14]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# Without a POS hint, the lemmatizer assumes everything is a NOUN:
print("studies ->", lemmatizer.lemmatize("studies"))         # study  (noun lookup works)
print("mice    ->", lemmatizer.lemmatize("mice"))            # mouse  (dictionary knowledge!)
print("running ->", lemmatizer.lemmatize("running"))         # running (treated as noun -> unchanged!)
print()
# Tell it 'running' is a VERB (pos='v') and it works:
print("running (as verb)  ->", lemmatizer.lemmatize("running", pos='v'))
print("ran     (as verb)  ->", lemmatizer.lemmatize("ran", pos='v'))      # irregular? No problem!
print("better  (as adj.)  ->", lemmatizer.lemmatize("better", pos='a'))   # the showpiece: better -> good

studies -> study
mice    -> mouse
running -> running

running (as verb)  -> run
ran     (as verb)  -> run
better  (as adj.)  -> good


`better → good` is the result no stemmer could ever produce — there is no suffix to chop; you need actual knowledge of English.

### Head-to-head: stemmer vs lemmatizer

In [15]:
words_pos = [("running", 'v'), ("ran", 'v'), ("studies", 'v'), ("better", 'a'),
             ("mice", 'n'), ("historical", 'a'), ("was", 'v')]

print(f"{'word':<12} {'stem':<12} {'lemma':<12}")
print("-" * 36)
for word, pos in words_pos:
    stem  = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos=pos)
    print(f"{word:<12} {stem:<12} {lemma:<12}")

word         stem         lemma       
------------------------------------
running      run          run         
ran          ran          run         
studies      studi        study       
better       better       good        
mice         mice         mouse       
historical   histor       historical  
was          wa           be          


The lemma column is always a real word; the stem column is faster to compute but messier.
**Need speed → stem. Need meaning → lemmatize.**

---
## 6. N-grams: capturing word order

Single tokens lose word order ("dog bites man" = "man bites dog" as a token *set*).
An **n-gram** is a sliding window of `n` consecutive tokens that puts local order back.

In [16]:
from nltk.util import ngrams

sentence = "I love natural language processing"
tokens = word_tokenize(sentence.lower())

unigrams = list(ngrams(tokens, 1))
bigrams  = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

print("Unigrams:", unigrams)
print("Bigrams: ", bigrams)
print("Trigrams:", trigrams)
print()
print(f"A {len(tokens)}-token sentence gives {len(bigrams)} bigrams and {len(trigrams)} trigrams (always n_tokens - n + 1).")

Unigrams: [('i',), ('love',), ('natural',), ('language',), ('processing',)]
Bigrams:  [('i', 'love'), ('love', 'natural'), ('natural', 'language'), ('language', 'processing')]
Trigrams: [('i', 'love', 'natural'), ('love', 'natural', 'language'), ('natural', 'language', 'processing')]

A 5-token sentence gives 4 bigrams and 3 trigrams (always n_tokens - n + 1).


Notice `('natural', 'language')` and `('language', 'processing')` — real multi-word concepts that single tokens completely miss.

### Mini-project: a tiny autocomplete

Your phone's word suggestion is, at heart, a **bigram frequency table**: given the last word, suggest the word that
most often follows it. Let's build one in ~15 lines.

In [17]:
from collections import Counter, defaultdict

# A tiny training corpus (in real life: millions of sentences)
corpus = """
I love natural language processing. Natural language processing is fun.
I love machine learning. Machine learning is powerful.
Natural language processing powers chatbots. I love deep learning.
Deep learning is amazing. Machine learning powers recommendation systems.
"""

# 1. Tokenize the whole corpus
tokens = word_tokenize(corpus.lower())
tokens = [t for t in tokens if t.isalpha()]   # keep only real words

# 2. Count every bigram
bigram_counts = defaultdict(Counter)
for w1, w2 in ngrams(tokens, 2):
    bigram_counts[w1][w2] += 1

# 3. Autocomplete = most frequent follower of the last word
def suggest_next(word, top_k=3):
    followers = bigram_counts[word.lower()]
    return followers.most_common(top_k)

for word in ["natural", "machine", "love", "is"]:
    print(f"After '{word}' -> {suggest_next(word)}")

After 'natural' -> [('language', 3)]
After 'machine' -> [('learning', 3)]
After 'love' -> [('natural', 1), ('machine', 1), ('deep', 1)]
After 'is' -> [('fun', 1), ('powerful', 1), ('amazing', 1)]


In [18]:
# We can even generate text by repeatedly taking the most likely next word:
def generate(start, length=6):
    word, result = start, [start]
    for _ in range(length):
        followers = bigram_counts[word].most_common(1)
        if not followers:
            break
        word = followers[0][0]
        result.append(word)
    return " ".join(result)

print("Generated:", generate("i"))
print("Generated:", generate("natural"))
print()
print("This is (very loosely!) the great-grandparent of GPT: predict the next token, repeat.")

Generated: i love natural language processing natural language
Generated: natural language processing natural language processing natural

This is (very loosely!) the great-grandparent of GPT: predict the next token, repeat.


---
## 7. The full pipeline — everything combined

Time to assemble all of today's tools into a single reusable function, exactly mirroring the pipeline slide:

**raw text → clean → tokenize → remove stop words → lemmatize → (optional) IDs**

In [19]:
import string

def preprocess(text, vocab=None, verbose=True):
    """Run the complete text processing pipeline on a string."""
    # Step 1: clean — lowercase
    cleaned = text.lower()

    # Step 2: tokenize
    tokens = word_tokenize(cleaned)

    # Step 3: drop punctuation tokens
    tokens = [t for t in tokens if t not in string.punctuation]

    # Step 4: remove stop words
    no_stops = [t for t in tokens if t not in stop_words]

    # Step 5: lemmatize (verbs first, then nouns — a simple but effective trick)
    lemmas = [lemmatizer.lemmatize(lemmatizer.lemmatize(t, pos='v'), pos='n') for t in no_stops]

    if verbose:
        print(f"Raw text     : {text}")
        print(f"Tokenized    : {tokens}")
        print(f"No stop words: {no_stops}")
        print(f"Lemmatized   : {lemmas}")

    # Step 6 (optional): map to vocabulary IDs
    if vocab is not None:
        ids = [vocab.get(t, vocab['[UNK]']) for t in lemmas]
        if verbose:
            print(f"Token IDs    : {ids}")
        return ids
    return lemmas

result = preprocess("The striped cats were running quickly!")

Raw text     : The striped cats were running quickly!
Tokenized    : ['the', 'striped', 'cats', 'were', 'running', 'quickly']
No stop words: ['striped', 'cats', 'running', 'quickly']
Lemmatized   : ['strip', 'cat', 'run', 'quickly']


In [20]:
# Try it on something messier — and on your own sentences!
print("=" * 60)
preprocess("Dr. Smith's students were studying tokenization, weren't they?")
print("=" * 60)
preprocess("The mice were better than the dogs at finding cheese!")

Raw text     : Dr. Smith's students were studying tokenization, weren't they?
Tokenized    : ['dr.', 'smith', "'s", 'students', 'were', 'studying', 'tokenization', 'were', "n't", 'they']
No stop words: ['dr.', 'smith', "'s", 'students', 'studying', 'tokenization', "n't"]
Lemmatized   : ['dr.', 'smith', "'s", 'student', 'study', 'tokenization', "n't"]
Raw text     : The mice were better than the dogs at finding cheese!
Tokenized    : ['the', 'mice', 'were', 'better', 'than', 'the', 'dogs', 'at', 'finding', 'cheese']
No stop words: ['mice', 'better', 'dogs', 'finding', 'cheese']
Lemmatized   : ['mouse', 'better', 'dog', 'find', 'cheese']


['mouse', 'better', 'dog', 'find', 'cheese']

Compare the first and last lines of each run: a messy human sentence in, a short list of clean, standardized
concepts out. **This is what "text preprocessing" means in practice** — and you just built the whole thing.

---
## 8. Bonus: a real BERT tokenizer (subwords in production)

Everything above is *classic* NLP. Modern models (BERT, GPT, LLaMA) skip stop word removal and lemmatization entirely —
their **subword tokenizers** handle word variation natively. Let's load BERT's actual tokenizer and watch it work.

> Requires the `transformers` library: `pip install transformers` (the tokenizer alone is lightweight — no PyTorch needed).

In [21]:
try:
    from transformers import AutoTokenizer

    bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    print(f"BERT vocabulary size: {bert_tok.vocab_size:,} tokens\n")

    for word in ["cat", "running", "unhappiness", "tokenization", "flurbification"]:
        pieces = bert_tok.tokenize(word)
        print(f"{word:<16} -> {pieces}")
except ImportError:
    print("transformers not installed — run: pip install transformers")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\DELL\miniconda3\envs\torch_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT vocabulary size: 30,522 tokens

cat              -> ['cat']
running          -> ['running']
unhappiness      -> ['un', '##ha', '##pp', '##iness']
tokenization     -> ['token', '##ization']
flurbification   -> ['flu', '##rb', '##ification']


Read the output carefully:

* **`cat`** is frequent → stays a single token.
* **`unhappiness`** → `un + ##ha + ##pp + ##iness` — the model still sees the negation prefix `un`!
* **`flurbification`** is a word we *just invented* — and BERT still tokenizes it from known pieces.
  **No `[UNK]`, no lost information.** Compare that to our word-level vocabulary in Section 2!

(`##` simply means "this piece continues the previous word".)

In [22]:
try:
    sentence = "The cats were running quickly!"

    encoded = bert_tok(sentence)
    tokens  = bert_tok.convert_ids_to_tokens(encoded["input_ids"])

    print("Sentence :", sentence)
    print("Tokens   :", tokens)
    print("Token IDs:", encoded["input_ids"])
    print()
    print("Note [CLS] and [SEP] — the special tokens from the vocabulary slide, added automatically.")
    print("Decoded back:", bert_tok.decode(encoded["input_ids"]))
except NameError:
    print("Run the previous cell first (transformers required).")

Sentence : The cats were running quickly!
Tokens   : ['[CLS]', 'the', 'cats', 'were', 'running', 'quickly', '!', '[SEP]']
Token IDs: [101, 1996, 8870, 2020, 2770, 2855, 999, 102]

Note [CLS] and [SEP] — the special tokens from the vocabulary slide, added automatically.
Decoded back: [CLS] the cats were running quickly! [SEP]


---
## 9. Exercises (try these yourself!)

1. **Tokenize your own text:** take a paragraph from any news article and count: how many word tokens? How many sentences? How many *unique* tokens?
2. **Stop word ratio:** what fraction of the tokens in your paragraph are stop words? (Typical English text: 40–50%!)
3. **Break the stemmer:** find two more examples of over-stemming or under-stemming with `PorterStemmer`.
4. **Improve the pipeline:** modify `preprocess()` so it *keeps* the word "not" — making it safe for sentiment analysis.
5. **Trigram autocomplete:** upgrade the autocomplete to use trigrams (predict from the last *two* words). Does the generated text improve?
6. **BERT exploration:** find a word that BERT splits into 4+ subword pieces. What's the weirdest split you can produce?

---
## Summary

| Concept | Tool we used | One-line takeaway |
|---|---|---|
| Tokenization | `word_tokenize`, `sent_tokenize` | Split text into units — punctuation & contractions need real rules |
| Vocabulary | our own `dict` | Tokens become IDs; unknown words become `[UNK]` |
| Stop words | `stopwords.words('english')` | Remove for search; keep for sentiment ("not good"!) |
| Stemming | `PorterStemmer` | Fast rule-based chopping; output may not be a word |
| Lemmatization | `WordNetLemmatizer` | Dictionary-correct base forms; needs part of speech |
| N-grams | `nltk.util.ngrams` | Sliding windows restore word order; power autocomplete |
| Subwords | `AutoTokenizer` (BERT) | Modern models solve OOV by splitting rare words into known pieces |

**Next session:** we take these clean tokens and turn them into *meaning* — text representation:
Bag of Words, TF-IDF, and our first steps toward word embeddings. See you there!